# 13 -- CNN RL from CNN SFT with parallel self-play (plan v5 Step 19, Derek Nov 27)

Step 18 (AlphaZero's training from CNN SFT pass 1, 19.3 GPU-h) ran with the GPU ~98% idle: the search is pure Python
and used one of the pod's 6 vCPUs. Step 19 changes **one thing**: each round's 128 self-play games are split over
`WORKERS` (5) processes, one per spare core (`search/parallel_selfplay.py`). Every other setting is unchanged.

Stages (`STAGES` from the shell, default `1,2,3`; every stage resumes where it stopped):
1. **Benchmark** -- one round from Step 18's r136 at 1, 3 and 5 workers: positions per hour and GPU utilization.
   **Gate: 5 workers >= 3x the positions per hour of 1**, else the notebook stops (nothing is trained). ~15 min.
2. **Train** -- from CNN SFT pass 1 until CNN SFT + RL = 19.3 GPU-h (run `az_b6c64_s0_cnnsft_p1_par5`). ~19 h.
3. **Rate** the final checkpoint on the full panel, with and without search (50 sims), next to Step 18's rows.

`SMOKE=1`: 2 tiny rounds with 2 workers in a `_smoke` folder, no benchmark or evals -- checks the code on the pod.


In [ ]:
# Setup (RunPod): the repo lives on the volume; artifacts under artifact_root (configs/base.yaml).
import os
_here = os.getcwd()
REPO_DIR = os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q -r requirements.txt

import json
import yaml
import torch
ROOT = yaml.safe_load(open("configs/base.yaml"))["artifact_root"]
from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- this needs a GPU pod"
print("GPU:", GPU)

## Settings
Network, RL and search settings from `configs/alphazero.yaml` (unchanged); the start is CNN SFT pass 1 (notebook 07).

In [ ]:
import dataclasses
from pathlib import Path
from eval.anchors import ResultsTable
from eval.panel import load_panel
from train.az_train import load_configs, run_name as az_run_name
from train.cnn_sft import CNNSFTConfig, gpu_hours_at as sft_gpu_hours_at, run_name as cnn_run_name

STAGES = {s.strip() for s in os.environ.get("STAGES", "1,2,3").split(",") if s.strip()}
SMOKE = os.environ.get("SMOKE", "0") == "1"
WORKERS = int(os.environ.get("WORKERS", "5"))      # one per spare vCPU on the 6-vCPU L4 pod
BENCH_WORKERS = [1, 3, WORKERS]
GATE_SPEEDUP = 3.0                                  # plan Step 19
RL_TOTAL_GPU_H = 19.3                               # as Step 18
SEARCH_SIMS = 50
eval_cfg = yaml.safe_load(open("configs/eval.yaml"))
GAMES, BATCH = eval_cfg["games_per_opponent"], eval_cfg["batch_games"]
PANEL = load_panel()
AZ_CFG, NET, MCTS, _ = load_configs("configs/alphazero.yaml")
CNN_CFG = CNNSFTConfig.from_dict(yaml.safe_load(open("configs/cnn_sft.yaml")))
CNN_RUN = cnn_run_name(CNN_CFG, NET)
CNN_DIR = os.path.join(ROOT, "checkpoints", "cnn_sft", CNN_RUN)
LABEL_CPU = json.load(open(os.path.join(ROOT, "data", "processed", CNN_CFG.dataset, "manifest.json")))["stockfish_cpu_seconds"] / 3600
STEP18_RUN, STEP18_ROUND = "az_b6c64_s0_cnnsft_p1", 136
results_path = os.path.join(ROOT, eval_cfg["results_file"])

AZ = dataclasses.replace(AZ_CFG, init_from=f"cnn_sft/{CNN_RUN}@p1", selfplay_workers=WORKERS,
                         tag=f"cnnsft_p1_par{WORKERS}" + ("_smoke" if SMOKE else ""))
AZ_MCTS = MCTS
if SMOKE:
    AZ = dataclasses.replace(AZ, games_per_round=4, max_plies=40, buffer_positions=5_000, checkpoint_every=1,
                             selfplay_workers=2)
    AZ_MCTS = dataclasses.replace(MCTS, simulations=8)
AZ_RUN = az_run_name(AZ, NET)
AZ_DIR = os.path.join(ROOT, "checkpoints", "alphazero", AZ_RUN)
print("stages:", sorted(STAGES), "| smoke:", SMOKE, "| workers:", AZ.selfplay_workers, "| run:", AZ_RUN)


## 1. Benchmark (gate)
The same network (Step 18's r136) plays one full round (128 games, 50 sims) serially and with 3 and 5 workers. Pool
start-up (~10 s per worker, once per run) is not timed. Positions per hour, not rounds, since game lengths vary.

In [ ]:
import time
from eval.gpu_util import GpuSampler, timed
from model.az_net import AZEvaluator
from search.mcts import play_games
from search.parallel_selfplay import ParallelSelfPlay
from train.az_train import load_checkpoint

if "1" in STAGES and not SMOKE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net = load_checkpoint(Path(ROOT) / "checkpoints" / "alphazero" / STEP18_RUN / "checkpoints" / f"round_{STEP18_ROUND:04d}", device)
    bench = {}
    with GpuSampler(interval=0.5) as sampler:
        for n in BENCH_WORKERS:
            kw = dict(opening_plies=AZ.opening_plies, max_plies=AZ.max_plies, seed=1)
            if n == 1:
                games, info = timed(sampler, lambda: play_games(AZEvaluator(net, device), AZ.games_per_round, MCTS, **kw))
            else:
                with ParallelSelfPlay(n, NET, MCTS, device) as pool:
                    games, info = timed(sampler, lambda: pool.play(net, AZ.games_per_round, **kw))
            pos = sum(len(g.moves) for g in games)
            bench[n] = {**info, "positions": pos, "positions_per_hour": round(pos / info["seconds"] * 3600)}
            print(f"{n} worker(s): {pos} positions in {info['seconds']:.0f} s = {bench[n]['positions_per_hour']:,}/h, "
                  f"GPU {info['util']:.1f}%")
    speedup = bench[WORKERS]["positions_per_hour"] / bench[1]["positions_per_hour"]
    json.dump(bench, open(os.path.join(ROOT, "results", "step19_benchmark.json"), "w"), indent=2)
    print(f"speedup at {WORKERS} workers: {speedup:.2f}x (gate {GATE_SPEEDUP}x)")
    if speedup < GATE_SPEEDUP:
        raise RuntimeError(f"Step 19 gate failed: {speedup:.2f}x < {GATE_SPEEDUP}x -- not training; discuss first")
    print("gate passed")
    del net
    torch.cuda.empty_cache()
else:
    print("benchmark skipped")


## 2. Train: CNN RL from CNN SFT pass 1, self-play on the workers
Until CNN SFT pass 1 + RL reaches 19.3 GPU-h. Resumable (rerun to continue). One row per round in `az_log.csv`.

In [ ]:
from train.az_train import list_checkpoints as az_checkpoints, train as train_az

if "2" in STAGES or SMOKE:
    init_h = sft_gpu_hours_at(CNN_DIR, 1)
    print(f"RL run: {AZ_RUN} | start: CNN SFT pass 1 ({init_h:.3f} GPU-h) | total budget {RL_TOTAL_GPU_H} GPU-h")
    summary = train_az(AZ, NET, AZ_MCTS, AZ_DIR, max_gpu_hours=None if SMOKE else RL_TOTAL_GPU_H,
                       max_rounds=2 if SMOKE else None, init_dir=Path(CNN_DIR) / "checkpoints" / "pass_0001",
                       init_gpu_hours=init_h, extra_info={"gpu": GPU, "step": 19})
    print(json.dumps(summary, indent=2))
    if SMOKE:
        print("smoke test passed")
else:
    print("training skipped")


## 3. Rate the final checkpoint, with and without search, next to Step 18

In [ ]:
import pandas as pd
from eval.anchors import build_reference_agents, close_agents
from eval.panel import extend
from train.az_train import gpu_hours_at as az_gpu_hours_at

if "3" in STAGES and not SMOKE:
    final = az_checkpoints(AZ_DIR)[-1][0]
    base = {"step": final, "seed": 0, "train_gpu_hours": round(az_gpu_hours_at(AZ_DIR, final), 4),
            "label_cpu_hours": LABEL_CPU, "dataset_size": json.load(open(os.path.join(CNN_DIR, "run_config.json")))["train_rows"]}
    note = f"CNN RL from CNN SFT pass 1, self-play on {WORKERS} processes (Step 19)"
    SPECS = [({"name": f"{AZ_RUN}_nosearch@r{final}", "kind": "alphazero", "run": AZ_RUN, "step": final, "simulations": 0},
              {**base, "run": f"alphazero/{AZ_RUN}_nosearch", "note": f"{note}, no search"}),
             ({"name": f"{AZ_RUN}@r{final}", "kind": "alphazero", "run": AZ_RUN, "step": final, "simulations": SEARCH_SIMS},
              {**base, "run": f"alphazero/{AZ_RUN}", "note": f"{note}, {SEARCH_SIMS}-sim search"})]
    results = ResultsTable(results_path)
    opponents = build_reference_agents(results, PANEL, ROOT)
    try:
        for spec, fields in SPECS:
            print(f"\n== {spec['name']}")
            extend(spec, results, ROOT, opponents, games_per_opponent=GAMES, batch_games=BATCH, fields=fields, gpu=GPU)
    finally:
        close_agents(opponents)
    t = ResultsTable(results_path)
    names = [f"{CNN_RUN}_nosearch@p1", f"{CNN_RUN}@p1", f"{CNN_RUN}_nosearch@p2", f"{CNN_RUN}@p2",
             f"{STEP18_RUN}_nosearch@r{STEP18_ROUND}", f"{STEP18_RUN}@r{STEP18_ROUND}", SPECS[0][0]["name"], SPECS[1][0]["name"]]
    display(pd.DataFrame([t.rows[n] for n in names if n in t.rows])[
        ["name", "rating", "ci_low", "ci_high", "games", "wins", "draws", "losses", "train_gpu_hours", "sec_per_move"]])
else:
    print("evals skipped")
